# 09 — Avansert RAG: Hybrid søk og reranking

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 08

**Hva du bygger:** Et forbedret RAG-system med hybrid søk (nøkkelord + semantisk) og reranking — de teknikkene SPK-annonsen nevner eksplisitt.

---

## Hvorfor er naivt RAG ikke nok?

```
Problem 1: Semantisk søk misser eksakte nøkkelord
  Q: "§ 23 i pensjonsloven"  →  Semantisk søk forstår ikke paragrafnummer

Problem 2: Topp-K gir ikke alltid beste chunk øverst
  Løsning: Reranking — en separat modell som sorterer på nytt

Problem 3: Vet ikke om svarene er gode
  Løsning: Evaluering med RAGAS-metrikker
```

In [ ]:
%pip install -q chromadb sentence-transformers rank-bm25 openai

In [ ]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from openai import OpenAI

embed_modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM_MODELL = "llama3.2"

# Dokumenter (samme som notatbok 08)
CHUNKS = [
    "AFP gir rett til tidligpensjon fra 62 år for offentlig ansatte.",
    "For å ha rett til AFP må du ha jobbet i offentlig sektor i minst tre år.",
    "AFP utbetales livsvarig og kan kombineres med alderspensjon.",
    "Alderspensjon fra SPK utbetales livsvarig fra fylte 67 år.",
    "Full pensjon krever 30 år i pensjonsordningen.",
    "Uførepensjon gis ved varig nedsatt arbeidsevne på minst 20 prosent.",
    "Uførepensjonen løper til du fyller 67 år, da konverteres den til alderspensjon.",
    "Barnepensjon utbetales til barn under 20 år ved forsørgers død.",
    "SPK-medlemmer kan søke boliglån til gunstig rente.",
    "Pensjonsopptjening skjer via innskudd fra arbeidsgiver over tid.",
]

print(f"{len(CHUNKS)} chunks klare.")

---

## Del 1: BM25 — Nøkkelordbasert søk

**BM25** er den klassiske søkemotoralgoritmen (brukt av Google, Elasticsearch). Den finner dokumenter basert på nøkkelordfrekvens — det semantisk søk ikke alltid klarer.

In [ ]:
# BM25 tokeniserer på ord
tokeniserte = [chunk.lower().split() for chunk in CHUNKS]
bm25 = BM25Okapi(tokeniserte)

def bm25_søk(spørsmål: str, topp_k: int = 3) -> list[tuple[float, str]]:
    tokens = spørsmål.lower().split()
    scorer = bm25.get_scores(tokens)
    topp   = np.argsort(scorer)[::-1][:topp_k]
    return [(scorer[i], CHUNKS[i]) for i in topp]

# Test: BM25 er god på eksakte ord
print("BM25-søk: 'AFP 62 år'")
for score, chunk in bm25_søk("AFP 62 år"):
    print(f"  [{score:.2f}] {chunk}")

---

## Del 2: Hybrid søk — kombiner BM25 + semantisk

In [ ]:
# Lag semantiske vektorer for alle chunks
chunk_vektorer = embed_modell.encode(CHUNKS)

def semantisk_søk(spørsmål: str, topp_k: int = 3) -> list[tuple[float, str]]:
    sv = embed_modell.encode([spørsmål])[0]
    likheter = [
        float(np.dot(sv, cv) / (np.linalg.norm(sv) * np.linalg.norm(cv)))
        for cv in chunk_vektorer
    ]
    topp = np.argsort(likheter)[::-1][:topp_k]
    return [(likheter[i], CHUNKS[i]) for i in topp]

def hybrid_søk(spørsmål: str, topp_k: int = 5, alfa: float = 0.5) -> list[tuple[float, str]]:
    """
    Kombiner BM25 og semantisk søk med Reciprocal Rank Fusion (RRF).
    alfa=1.0 → kun semantisk, alfa=0.0 → kun BM25
    """
    bm25_res = bm25_søk(spørsmål, topp_k=len(CHUNKS))
    sem_res  = semantisk_søk(spørsmål, topp_k=len(CHUNKS))
    
    # Normaliser scorer til 0-1
    def normaliser(resultater):
        scorer = [s for s, _ in resultater]
        min_s, max_s = min(scorer), max(scorer)
        if max_s == min_s:
            return [(0.5, c) for _, c in resultater]
        return [((s - min_s) / (max_s - min_s), c) for s, c in resultater]
    
    bm25_norm = {c: s for s, c in normaliser(bm25_res)}
    sem_norm  = {c: s for s, c in normaliser(sem_res)}
    
    # Vektet sum
    kombinert = {}
    for chunk in CHUNKS:
        kombinert[chunk] = (
            alfa * sem_norm.get(chunk, 0) +
            (1 - alfa) * bm25_norm.get(chunk, 0)
        )
    
    sortert = sorted(kombinert.items(), key=lambda x: x[1], reverse=True)
    return [(s, c) for c, s in sortert[:topp_k]]

# Sammenlign metodene
spørsmål = "Kan jeg gå av tidlig som offentlig ansatt?"
print(f"Spørsmål: {spørsmål}\n")

print("Kun semantisk:")
for s, c in semantisk_søk(spørsmål, 2): print(f"  [{s:.3f}] {c}")

print("\nKun BM25:")
for s, c in bm25_søk(spørsmål, 2): print(f"  [{s:.3f}] {c}")

print("\nHybrid (50/50):")
for s, c in hybrid_søk(spørsmål, 2): print(f"  [{s:.3f}] {c}")

---

## Del 3: Reranking

**Reranking** bruker en kraftigere modell (cross-encoder) til å score par av (spørsmål, chunk) direkte — mye mer presist enn vektorsøk alene, men for tregt til å kjøre på alle dokumenter.

**Mønster:** Hent mange kandidater raskt → rerank de beste

In [ ]:
# Cross-encoder: gratis, lokalt, liten (~70 MB)
# Støtter mange språk inkl. norsk
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def søk_og_rerank(spørsmål: str, kandidater: int = 6, topp_k: int = 3) -> list:
    # Steg 1: Hent mange kandidater raskt med hybrid søk
    kandidat_chunks = [c for _, c in hybrid_søk(spørsmål, topp_k=kandidater)]
    
    # Steg 2: Rerank med cross-encoder
    par    = [(spørsmål, chunk) for chunk in kandidat_chunks]
    scorer = reranker.predict(par)
    
    # Sorter på ny score
    reranket = sorted(zip(scorer, kandidat_chunks), reverse=True)
    return reranket[:topp_k]

print("Etter reranking:")
for score, chunk in søk_og_rerank("Tidligpensjon offentlig ansatt"):
    print(f"  [{score:.3f}] {chunk}")

---

## Del 4: Enkel RAG-evaluering

Hvordan vet du om RAG-systemet ditt er godt? Du måler det.

In [ ]:
def evaluer_retrieval(testset: list[dict]) -> dict:
    """
    Mål retrieval-kvalitet: hit-rate@k
    (Fant vi det relevante dokumentet i topp-k?)
    """
    treff = 0
    for test in testset:
        resultater = [c for _, c in søk_og_rerank(test["spørsmål"], topp_k=3)]
        if any(test["forventet_nøkkelord"] in r for r in resultater):
            treff += 1
    return {"hit_rate@3": treff / len(testset), "treff": treff, "totalt": len(testset)}

testset = [
    {"spørsmål": "Når kan jeg ta ut AFP?",              "forventet_nøkkelord": "62 år"},
    {"spørsmål": "Hva er kravet for AFP?",              "forventet_nøkkelord": "tre år"},
    {"spørsmål": "Uførepensjon sats",                   "forventet_nøkkelord": "20 prosent"},
    {"spørsmål": "Hva skjer med pensjonen ved 67 år?",  "forventet_nøkkelord": "67 år"},
]

metrics = evaluer_retrieval(testset)
print(f"Hit-rate@3: {metrics['hit_rate@3']:.0%}  ({metrics['treff']}/{metrics['totalt']} spørsmål besvart riktig)")

---

## Oppsummering

| Teknikk | Hva det løser | Verktøy |
|---------|--------------|--------|
| BM25 | Eksakte nøkkelord, tall, navn | `rank-bm25` |
| Semantisk søk | Mening og synonymer | `sentence-transformers` |
| Hybrid søk | Begge deler | Kombiner BM25 + semantisk |
| Reranking | Sorter kandidater presist | `CrossEncoder` |
| Hit-rate@k | Mål om retrieval fungerer | Egne testdata |

---

## Hva er neste steg?

**Neste: `10_ai_agents.ipynb`** — RAG er passivt (hent og svar). Agenter er aktive — de kan bruke verktøy, ta beslutninger og gjøre flere steg for å løse komplekse oppgaver.